# Member 4 — Normalization / Scaling

**Technique:** Standardize handcrafted features with `StandardScaler` (zero mean, unit variance) and compare with Min–Max scaling.

## Why this dataset needs it
Color histogram bins and channel statistics sit on different numeric ranges. Distance-based and regularized models (KNN, SVM, logistic regression) are sensitive to unscaled features.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

clean_path = OUT / "m3_cleaned_no_outliers.csv"
if clean_path.exists():
    meta = pd.read_csv(clean_path)
else:
    meta, _ = audit_images(RAW)
    meta, _ = remove_exact_duplicates(meta)
    print("Warning: m3 output missing; using audited unique images.")

# Use a stratified sample for interactive demo speed (full run still OK on ~1k images)
sample = stratified_sample(meta, 120)

X, y, kept = extract_feature_matrix(RAW, sample)
print("Feature matrix:", X.shape, "| classes:", dict(zip(*np.unique(y, return_counts=True))))

scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X)

scaler_mm = MinMaxScaler()
X_mm = scaler_mm.fit_transform(X)

# Persist for Members 5–6 / group pipeline
np.savez_compressed(
    OUT / "m4_scaled_features.npz",
    X_raw=X,
    X_standard=X_std,
    X_minmax=X_mm,
    y=y,
    feature_dim=np.array([X.shape[1]]),
)
kept.to_csv(OUT / "m4_feature_sample_index.csv", index=False)
print("Saved scaled feature matrices to results/outputs/m4_scaled_features.npz")


## EDA visualization — feature distribution before vs after StandardScaler


In [ ]:
# Compare first feature column distribution
feat_idx = 0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X[:, feat_idx], bins=30, color="#1f77b4", alpha=0.85)
axes[0].set_title(f"Raw feature[{feat_idx}]")
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Frequency")

axes[1].hist(X_std[:, feat_idx], bins=30, color="#ff7f0e", alpha=0.85)
axes[1].set_title(f"StandardScaler feature[{feat_idx}]")
axes[1].set_xlabel("Z-score")

fig.tight_layout()
fig.savefig(VIZ / "m4_scaling_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

print("Raw mean/std:", float(X[:, feat_idx].mean()), float(X[:, feat_idx].std()))
print("Scaled mean/std:", float(X_std[:, feat_idx].mean()), float(X_std[:, feat_idx].std()))
print("Interpretation: after StandardScaler, features centre near 0 with comparable variance — safer for distance-based learners.")


## Viva talking points
1. Difference between StandardScaler and MinMaxScaler.
2. Why image-derived histograms need scaling.
3. Interpret the before/after histogram.
